# Fractals: the Mandelbrot and Julia Sets

Take a complex number $c$, start at $z = 0$, and repeat one instruction: square and add $c$. Either the numbers stay in a bounded region forever or they fly off to infinity. Colour each $c$ by what happens and you get the **Mandelbrot set**, the most famous picture in mathematics: a cardioid with bulbs attached, decorated with spirals, filaments and tiny copies of itself at every scale. Fix $c$ instead and vary the starting point, and you get the **Julia sets**, a whole family of fractals indexed by $c$.

This notebook explains the geometry of $z \mapsto z^2 + c$, proves the **escape radius 2** criterion that makes the pictures computable, renders the sets with vectorized numpy, derives the formula of the **main cardioid**, and connects the two families through one theorem: the Julia set of $c$ is connected exactly when $c$ lies in the Mandelbrot set.

This is notebook 05 of module 07 (**Applications**) in the **Learning Complex Numbers** series.

## 1. Introduction

Every notebook so far has been about *linear* behaviour: rotations, linear ODEs, filters. Squaring is the simplest nonlinear operation, and iterating it already produces complexity without limit. Yet each single step is plain complex arithmetic you met in module 01:

- **Squaring** squares the modulus and **doubles the argument**: $\left(re^{i\theta}\right)^2 = r^2e^{2i\theta}$. Points inside the unit circle are pulled in, points outside are pushed out, and every angle is doubled.
- **Adding $c$** translates the whole plane by the vector $c$.

The tension between "push outward" and "shift sideways" decides whether an orbit escapes. Gaston Julia and Pierre Fatou studied these iterations around 1918 without computers; Benoît Mandelbrot's computer pictures of 1980 made the sets famous. Newton's method (notebook 02/03) produced fractals for the same underlying reason: iterating a complex rational map.

**Prerequisites**

- The complex plane and the modulus — [The complex plane](../00_foundations/04_the_complex_plane.ipynb), [Conjugate, modulus and division](../00_foundations/03_conjugate_modulus_and_division.ipynb).
- Squaring as "square the modulus, double the angle" — [Multiplication is rotation and scaling](../01_polar_form_and_euler/02_multiplication_is_rotation_and_scaling.ipynb).
- Iterating complex maps and fractal basins — [Newton's method in the complex plane](../02_polynomials_and_roots/03_newtons_method_in_the_complex_plane.ipynb).
- Complex derivatives (for the stability of fixed points) — [The complex derivative and Cauchy–Riemann](../05_complex_differentiation/02_the_complex_derivative_and_cauchy_riemann.ipynb).

After completing this notebook you will be able to:

- Compute orbits of $f_c(z) = z^2 + c$ and describe them geometrically.
- Prove that an orbit escapes once $|z_n| > \max(2, |c|)$, and why $|z_n| > 2$ is the exact test for the Mandelbrot set.
- Render the Mandelbrot set and Julia sets with vectorized escape-time code.
- Derive the main cardioid $c = \frac{\mu}{2} - \frac{\mu^2}{4}$ and the period-2 disk $|c + 1| < \frac14$ from attracting cycles.
- State the connectedness dichotomy linking $c \in M$ to the shape of the Julia set.

## 2. Theory

### 2.1 Orbits

For a fixed $c \in \mathbb{C}$ let $f_c(z) = z^2 + c$. The **orbit** of a starting point $z_0$ is $z_0, z_1 = f_c(z_0), z_2 = f_c(z_1), \dots$. Typical behaviours: escape to $\infty$; convergence to a fixed point ($f_c(z^*) = z^*$); convergence to a cycle of period $p$; or bounded but chaotic wandering.

### 2.2 Two families of sets

- The **filled Julia set** of $c$: $\ K_c = \{z_0 \in \mathbb{C} : \text{the orbit of } z_0 \text{ under } f_c \text{ is bounded}\}$. Its boundary $J_c = \partial K_c$ is the **Julia set**. Here $c$ is fixed and the starting point varies.
- The **Mandelbrot set**: $\ M = \{c \in \mathbb{C} : \text{the orbit of } 0 \text{ under } f_c \text{ is bounded}\}$. Here the starting point is fixed at $z_0 = 0$ and $c$ varies.

Why start at $0$? It is the only **critical point** of $f_c$ ($f_c'(z) = 2z = 0$), and the fate of the critical orbit controls the whole Julia set (section 2.5).

Two easy examples: for $c = 0$, $z_n = z_0^{2^n}$, so $K_0$ is the closed unit disk and $J_0$ the unit circle. For $c = -2$, $K_{-2} = J_{-2}$ is the segment $[-2, 2]$ (substitute $z = 2\cos\theta$: then $z^2 - 2 = 2\cos 2\theta$).

### 2.3 The escape radius

**Lemma.** If $|z| > 2$ and $|z| \ge |c|$, then $|f_c(z)| > |z|$, and the orbit of $z$ tends to $\infty$.

*Proof.* By the reverse triangle inequality,

$$
|z^2 + c| \ge |z|^2 - |c| \ge |z|^2 - |z| = |z|\,(|z| - 1).
$$

Write $|z| = 2 + \varepsilon$ with $\varepsilon > 0$. Then $|f_c(z)| \ge (1 + \varepsilon)|z| > |z| \ge |c|$, so the hypothesis holds again for $f_c(z)$, now with an even larger $\varepsilon$. By induction $|z_n| \ge (1 + \varepsilon)^n|z_0| \to \infty$. $\blacksquare$

**Consequences.**

1. **Julia sets:** $K_c$ lies in the disk $|z| \le R_c = \max(2, |c|)$; any orbit that leaves it escapes.
2. **Mandelbrot set:** if $|c| > 2$, then $z_1 = c$ already satisfies the lemma, so $c \notin M$. If $|c| \le 2$ and some $|z_n| > 2$, then $|z_n| > 2 \ge |c|$ and the orbit escapes. Hence

$$
\boxed{\,c \in M \iff |z_n| \le 2 \ \text{for all } n\,} \qquad\text{and}\qquad M \subset \{|c| \le 2\}.
$$

This is what makes the pictures computable: iterate up to a maximum count $N$, and stop as soon as $|z_n| > 2$. Points that never exceed $2$ within $N$ steps are *probably* in $M$ — the picture is an approximation from outside, and higher $N$ reveals finer detail near the boundary.

The **escape time** (the first $n$ with $|z_n| > 2$) is a natural colouring: points close to $M$ take long to escape.

### 2.4 Attracting fixed points and the main cardioid

A fixed point $z^*$ is **attracting** if $|f_c'(z^*)| = |2z^*| < 1$: nearby points are pulled in, because $f_c(z^* + h) \approx z^* + 2z^*h$ shrinks the offset $h$ by the factor $|2z^*|$. (This is the complex derivative acting as a local rotation-and-scaling.)

Parametrise by the **multiplier** $\mu = 2z^*$. The fixed-point equation $z^{*2} + c = z^*$ then says

$$
c = z^* - z^{*2} = \frac{\mu}{2} - \frac{\mu^2}{4}.
$$

As $\mu$ ranges over the open unit disk, $c$ fills the region of parameters with an attracting fixed point. Its boundary, $\mu = e^{it}$, is the **main cardioid**:

$$
c(t) = \frac{e^{it}}{2} - \frac{e^{2it}}{4}, \qquad t \in [0, 2\pi).
$$

At $t = 0$ it has a cusp at $c = \frac14$; at $t = \pi$ it reaches $c = -\frac12 - \frac14 = -\frac34$. When $f_c$ has an attracting fixed point, the critical orbit is attracted to it (a theorem of Fatou), so the cardioid lies inside $M$.

### 2.5 The period-2 bulb

A 2-cycle $\{z_1, z_2\}$ solves $f_c(f_c(z)) = z$ but not $f_c(z) = z$. Dividing $f_c(f_c(z)) - z$ by $f_c(z) - z$ leaves $z^2 + z + c + 1 = 0$, whose roots satisfy $z_1z_2 = c + 1$. The multiplier of the cycle is $f_c'(z_1)f_c'(z_2) = 4z_1z_2 = 4(c + 1)$, so the 2-cycle attracts exactly when

$$
|c + 1| < \tfrac14,
$$

the disk of radius $\frac14$ centred at $-1$, touching the cardioid at $-\frac34$. Every bulb of $M$ corresponds to an attracting cycle of some period.

### 2.6 The fundamental dichotomy

**Theorem** (Fatou, Julia). For $f_c(z) = z^2 + c$:

- if $c \in M$ (the critical orbit is bounded), the Julia set $J_c$ and the filled set $K_c$ are **connected**;
- if $c \notin M$, they are a **Cantor set** ("Fatou dust"): totally disconnected, uncountably many points, no area.

So the Mandelbrot set is a map of the Julia sets: inside, connected shapes (circles, basilicas, rabbits, dendrites); outside, dust. It is itself connected (Douady and Hubbard, 1982), and its boundary has Hausdorff dimension $2$ (Shishikura, 1998).

Two symmetries are immediate: $\overline{f_c(z)} = f_{\bar c}(\bar z)$ makes $M$ symmetric about the real axis, and $f_c(-z) = f_c(z)$ makes every $K_c$ symmetric under $z \mapsto -z$.

## 3. Worked Examples

### Example 1 — three orbits of $0$

- $c = -1$: $0 \to -1 \to 0 \to -1 \to \cdots$, a 2-cycle. Bounded, so $-1 \in M$ (indeed $|c + 1| = 0 < \frac14$: centre of the period-2 bulb).
- $c = i$: $0 \to i \to i^2 + i = -1 + i \to (-1 + i)^2 + i = -2i + i = -i \to (-i)^2 + i = -1 + i \to -i \to \cdots$. After two steps it cycles between $-1 + i$ and $-i$: bounded, so $i \in M$ (a tip of a filament).
- $c = 1$: $0 \to 1 \to 2 \to 5 \to 26 \to 677 \to \cdots$. At $z_3 = 5 > 2$ the escape test fires: $1 \notin M$.

### Example 2 — an attracting fixed point for $c = 0.2$

Fixed points: $z^2 - z + 0.2 = 0$, so $z^* = \frac{1 \pm \sqrt{1 - 0.8}}{2} = \frac{1 \pm 0.4472}{2}$, i.e. $0.2764$ or $0.7236$. Multipliers $|2z^*| = 0.553$ (attracting) and $1.447$ (repelling). With an attracting fixed point, $c = 0.2$ is in the main cardioid and the orbit of $0$ converges to $0.2764$.

### Example 3 — points of the cardioid

$c(t) = \frac{e^{it}}{2} - \frac{e^{2it}}{4}$:

- $t = 0$: $c = \frac12 - \frac14 = \frac14$ (the cusp).
- $t = \pi$: $c = -\frac12 - \frac14 = -\frac34$ (where the period-2 disk is attached).
- $t = \frac{\pi}{2}$: $c = \frac{i}{2} - \frac{-1}{4} = \frac14 + \frac{i}{2}$.

### Example 4 — the escape lemma in action

$c = 0.5$: $z_1 = 0.5$, $z_2 = 0.75$, $z_3 = 1.0625$, $z_4 = 1.6289$, $z_5 = 3.1533 > 2$. From here $|z_6| \ge 3.1533 \cdot 2.1533 = 6.79$ (actual value $10.44$), and the growth only accelerates. So $0.5 \notin M$ — consistent with the cusp at $\frac14$ being the right-most point of $M$ on the real axis.

### Example 5 — the Julia set of $c = 0$

$z_n = z_0^{2^n}$ has modulus $|z_0|^{2^n}$: it tends to $0$ if $|z_0| < 1$, to $\infty$ if $|z_0| > 1$, and stays on the unit circle if $|z_0| = 1$ (where it doubles the angle each step — chaotically). So $K_0$ is the closed unit disk, $J_0$ the unit circle.

## 4. Python Examples

### 4.1 Orbits with plain complex numbers

In [ ]:
def orbit(c, z0=0j, steps=10, escape_radius=2.0):
    points = [complex(z0)]
    z = complex(z0)
    for _ in range(steps):
        z = z * z + c
        points.append(z)
        if abs(z) > max(escape_radius, abs(c)) and abs(z) > 1e6:
            break
    return points


for c in [-1, 1j, 1, 0.2, 0.5]:
    pts = orbit(c, steps=7)
    print(f"c = {c}:", ", ".join(f"{p.real:.4g}{p.imag:+.4g}j" for p in pts))

### 4.2 Fixed points and their multipliers

In [ ]:
import numpy as np
import sympy as sp

for c in [0.2, -0.5, -1.1, 0.3 + 0.6j]:
    fixed_points = np.roots([1, -1, c])               # z^2 - z + c = 0
    multipliers = np.abs(2 * fixed_points)
    print(f"c = {c}: fixed points {np.round(fixed_points, 4)}, |multipliers| {np.round(multipliers, 4)}, "
          f"attracting fixed point: {bool(np.min(multipliers) < 1)},  |c + 1| = {abs(c + 1):.3f}")

# Symbolic: the 2-cycle equation and its multiplier
z, c = sp.symbols("z c")
f = lambda w: w**2 + c
quotient, remainder = sp.div(sp.expand(f(f(z)) - z), sp.expand(f(z) - z), z)
print("f(f(z)) - z = (f(z) - z) *", quotient, "  remainder", remainder)
z1, z2 = sp.solve(quotient, z)
print("multiplier 4 z1 z2 =", sp.simplify(4 * z1 * z2))

### 4.3 Vectorized escape time

Iterating every pixel in a Python loop would be slow. With numpy we iterate a whole grid of $c$ values at once, keeping a boolean mask of the points that are still bounded so that escaped points are frozen (this also avoids overflow warnings). The function returns the escape count, with `max_iter` for points that never escaped.

In [ ]:
import time

import numpy as np


def escape_time(c, z0, max_iter):
    # Escape counts for z -> z^2 + c (c and z0 arrays of the same shape)
    z = z0.astype(complex).copy()
    c = np.broadcast_to(c, z.shape).astype(complex)
    counts = np.full(z.shape, max_iter, dtype=int)
    radius = np.maximum(2.0, np.abs(c))
    active = np.ones(z.shape, dtype=bool)
    for n in range(max_iter):
        z[active] = z[active] ** 2 + c[active]
        escaped = active & (np.abs(z) > radius)
        counts[escaped] = n + 1
        active &= ~escaped
        if not active.any():
            break
    return counts


def complex_grid(re_min, re_max, im_min, im_max, width, height):
    re = np.linspace(re_min, re_max, width)
    im = np.linspace(im_min, im_max, height)
    return re[None, :] + 1j * im[:, None]


start = time.perf_counter()
C = complex_grid(-2.2, 0.8, -1.3, 1.3, 600, 520)
mandelbrot_counts = escape_time(C, np.zeros_like(C), max_iter=200)
elapsed = time.perf_counter() - start
inside = mandelbrot_counts == 200
pixel_area = (3.0 / 599) * (2.6 / 519)
print(f"600 x 520 grid, 200 iterations: {elapsed:.2f} s")
print(f"fraction of pixels that never escaped: {inside.mean():.4f}")
print(f"area estimate of M: {inside.sum() * pixel_area:.3f}  (best known value about 1.506)")

### 4.4 Testing the cardioid and the bulb

Every grid point strictly inside the cardioid or the period-2 disk should never escape. To test "inside the cardioid" for a given $c$ we invert $c = \frac{\mu}{2} - \frac{\mu^2}{4}$: $\mu = 1 - \sqrt{1 - 4c}$ (the root with $|\mu|$ smaller), and check $|\mu| < 1$.

In [ ]:
import numpy as np


def in_main_cardioid(c):
    mu = 1 - np.sqrt(1 - 4 * np.asarray(c, dtype=complex))
    return np.abs(mu) < 1


def in_period_two_disk(c):
    return np.abs(np.asarray(c) + 1) < 0.25


def escape_time(c, max_iter):
    z = np.zeros_like(c)
    counts = np.full(c.shape, max_iter)
    active = np.ones(c.shape, dtype=bool)
    for n in range(max_iter):
        z[active] = z[active] ** 2 + c[active]
        escaped = active & (np.abs(z) > 2)
        counts[escaped] = n + 1
        active &= ~escaped
    return counts


rng = np.random.default_rng(0)
samples = rng.uniform(-2, 2, 200000) + 1j * rng.uniform(-2, 2, 200000)
counts = escape_time(samples, max_iter=300)
cardioid = in_main_cardioid(samples)
bulb = in_period_two_disk(samples)
print("points in cardioid:", cardioid.sum(), "  escaped among them:", (counts[cardioid] < 300).sum())
print("points in 2-bulb  :", bulb.sum(), "  escaped among them:", (counts[bulb] < 300).sum())
print("share of the bounded samples covered by cardioid + bulb:",
      round(((cardioid | bulb) & (counts == 300)).sum() / (counts == 300).sum(), 3))
print("cardioid test at 0.25+0.5j (boundary, |mu| = 1):", np.round(np.abs(1 - np.sqrt(1 - 4 * (0.25 + 0.5j))), 12))

The cardioid (area $\frac{3\pi}{8} \approx 1.178$) and the period-2 disk (area $\frac{\pi}{16} \approx 0.196$) alone account for about 90% of the area of $M$; the rest is the countless smaller bulbs and filaments.

## 5. Visualizations

### 5.1 Orbits of $0$ in the plane

Four values of $c$: converging to an attracting fixed point (spiralling in), a 2-cycle, a 3-cycle (the "rabbit" parameter), and an escaping orbit. Each orbit is drawn as connected dots starting at $z_0 = 0$; the dashed circle is the escape radius $2$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cases = [(-0.3 + 0.4j, "fixed point"), (-1.0 + 0.0j, "2-cycle"), (-0.1226 + 0.7449j, "3-cycle"), (0.3 + 0.6j, "escapes")]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
circle = 2 * np.exp(1j * np.linspace(0, 2 * np.pi, 300))
for ax, (c, label) in zip(axes, cases):
    z, points = 0j, [0j]
    for _ in range(60):
        z = z * z + c
        points.append(z)
        if abs(z) > 2.5:
            break
    points = np.array(points)
    ax.plot(circle.real, circle.imag, "--", color="gray", linewidth=1)
    ax.plot(points.real, points.imag, "-", color="lightsteelblue", linewidth=1)
    ax.scatter(points.real, points.imag, c=np.arange(len(points)), cmap="viridis", s=25, zorder=3)
    ax.plot(c.real, c.imag, "r+", markersize=12, markeredgewidth=2)
    ax.axhline(0, color="black", linewidth=0.6)
    ax.axvline(0, color="black", linewidth=0.6)
    ax.set_xlim(-2.7, 2.7)
    ax.set_ylim(-2.7, 2.7)
    ax.set_aspect("equal")
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.set_title(f"c = {c.real:g}{c.imag:+g}i: {label}")
plt.suptitle("Orbits of 0 under $z \\mapsto z^2 + c$ (colour = step, red + = c)", y=1.02)
plt.tight_layout()
plt.show()

### 5.2 How fast escape happens

$|z_n|$ on a log scale for a few parameters outside $M$. Below the line $|z| = 2$ an orbit can loiter for a long time (longer the closer $c$ is to $M$), but once it crosses $2$ the modulus roughly squares at every step: doubly exponential growth, exactly as the lemma predicts.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(9, 5))
for c in [1.0, 0.5, 0.3 + 0.6j, 0.26, -0.75 + 0.05j]:
    z, moduli = 0j, []
    for _ in range(200):
        z = z * z + c
        moduli.append(abs(z))
        if abs(z) > 1e30:
            break
    plt.semilogy(range(1, len(moduli) + 1), moduli, "o-", markersize=3, label=f"c = {c}")
plt.axhline(2, color="black", linestyle="--", label="escape radius 2")
plt.xlabel("n")
plt.ylabel("$|z_n|$")
plt.title("Orbits of 0: a long wait below 2, then explosive growth")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

### 5.3 The Mandelbrot set, with the cardioid and the period-2 disk

An 800 × 700 escape-time image (colour = log of escape time, black = never escaped within 250 steps). The cyan curve is the cardioid $c(t) = \frac{e^{it}}{2} - \frac{e^{2it}}{4}$, the white circle the disk $|c + 1| = \frac14$, and the dashed circle $|c| = 2$ bounds the whole set.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def escape_time(c, max_iter):
    z = np.zeros_like(c)
    counts = np.full(c.shape, max_iter)
    active = np.ones(c.shape, dtype=bool)
    for n in range(max_iter):
        z[active] = z[active] ** 2 + c[active]
        escaped = active & (np.abs(z) > 2)
        counts[escaped] = n + 1
        active &= ~escaped
    return counts


max_iter = 250
re = np.linspace(-2.25, 0.75, 800)
im = np.linspace(-1.3, 1.3, 700)
C = re[None, :] + 1j * im[:, None]
counts = escape_time(C, max_iter)
image = np.ma.masked_where(counts == max_iter, np.log(counts))

fig, ax = plt.subplots(figsize=(10, 8.5))
cmap = plt.get_cmap("twilight_shifted").copy()
cmap.set_bad("black")
ax.imshow(image, extent=[re[0], re[-1], im[0], im[-1]], origin="lower", cmap=cmap)

t = np.linspace(0, 2 * np.pi, 500)
cardioid = np.exp(1j * t) / 2 - np.exp(2j * t) / 4
disk = -1 + 0.25 * np.exp(1j * t)
big_circle = 2 * np.exp(1j * t)
ax.plot(cardioid.real, cardioid.imag, color="cyan", linewidth=1.5, label="main cardioid")
ax.plot(disk.real, disk.imag, color="white", linewidth=1.5, label="$|c + 1| = 1/4$")
ax.plot(big_circle.real, big_circle.imag, "--", color="orange", linewidth=1, label="$|c| = 2$")
ax.set_xlim(re[0], re[-1])
ax.set_ylim(im[0], im[-1])
ax.set_aspect("equal")
ax.set_xlabel("Re c")
ax.set_ylabel("Im c")
ax.set_title("The Mandelbrot set M")
ax.legend(loc="upper left")
plt.show()

### 5.4 Zooming in: the seahorse valley

The region between the cardioid and the period-2 disk, magnified about 30 times. More iterations are needed near the boundary, where escape is slow.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def escape_time(c, max_iter):
    z = np.zeros_like(c)
    counts = np.full(c.shape, max_iter)
    active = np.ones(c.shape, dtype=bool)
    for n in range(max_iter):
        z[active] = z[active] ** 2 + c[active]
        escaped = active & (np.abs(z) > 2)
        counts[escaped] = n + 1
        active &= ~escaped
    return counts


max_iter = 400
center, half_width = -0.745 + 0.11j, 0.05
re = np.linspace(center.real - half_width, center.real + half_width, 600)
im = np.linspace(center.imag - half_width, center.imag + half_width, 600)
C = re[None, :] + 1j * im[:, None]
counts = escape_time(C, max_iter)
image = np.ma.masked_where(counts == max_iter, np.log(counts))

fig, ax = plt.subplots(figsize=(8, 8))
cmap = plt.get_cmap("magma").copy()
cmap.set_bad("black")
ax.imshow(image, extent=[re[0], re[-1], im[0], im[-1]], origin="lower", cmap=cmap)
ax.set_aspect("equal")
ax.set_xlabel("Re c")
ax.set_ylabel("Im c")
ax.set_title("Seahorse valley around c = -0.745 + 0.11i")
plt.show()

### 5.5 A gallery of Julia sets, and where their $c$ lives in $M$

Six filled Julia sets $K_c$ (black = bounded orbits), each $400 \times 400$. The first five parameters lie in $M$ and give connected sets: the disk ($c = 0$), the basilica ($c = -1$), Douady's rabbit (an attracting 3-cycle), a spiralling set for $c = -0.5 + 0.5i$ near the edge of the cardioid, and a set with an attracting 4-cycle from the bulb at $c = -1.3$. The last parameter lies outside $M$ and the set shatters into dust; the coloured escape times show its ghost. The small map marks each $c$ on the Mandelbrot set.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def escape_time(z0, c, max_iter):
    z = z0.copy()
    counts = np.full(z.shape, max_iter)
    active = np.ones(z.shape, dtype=bool)
    radius = max(2.0, abs(c))
    for n in range(max_iter):
        z[active] = z[active] ** 2 + c
        escaped = active & (np.abs(z) > radius)
        counts[escaped] = n + 1
        active &= ~escaped
    return counts


def in_mandelbrot(c, max_iter=1000):
    z = 0j
    for _ in range(max_iter):
        z = z * z + c
        if abs(z) > 2:
            return False
    return True


parameters = [0j, -1 + 0j, -0.1226 + 0.7449j, -0.5 + 0.5j, -1.3 + 0j, 0.3 + 0.6j]
names = ["disk", "basilica", "Douady rabbit", "spiralling", "period-4 bulb", "dust"]
max_iter = 200
grid = np.linspace(-1.7, 1.7, 400)
Z0 = grid[None, :] + 1j * grid[:, None]

fig, axes = plt.subplots(2, 4, figsize=(18, 9), gridspec_kw={"width_ratios": [1, 1, 1, 1.15]})
panel_axes = [axes[0, 0], axes[0, 1], axes[0, 2], axes[1, 0], axes[1, 1], axes[1, 2]]
cmap = plt.get_cmap("viridis").copy()
cmap.set_bad("black")
for ax, c, name in zip(panel_axes, parameters, names):
    counts = escape_time(Z0, c, max_iter)
    image = np.ma.masked_where(counts == max_iter, np.log(counts))
    ax.imshow(image, extent=[-1.7, 1.7, -1.7, 1.7], origin="lower", cmap=cmap)
    membership = "c in M" if in_mandelbrot(c) else "c not in M"
    ax.set_title(f"c = {c.real:g}{c.imag:+g}i  ({membership}) {name}", fontsize=10)
    ax.set_aspect("equal")
    ax.set_xlabel("Re z")
    ax.set_ylabel("Im z")

gs = axes[0, 3].get_gridspec()
axes[0, 3].remove()
axes[1, 3].remove()
map_ax = fig.add_subplot(gs[:, 3])
re = np.linspace(-2.1, 0.7, 420)
im = np.linspace(-1.3, 1.3, 390)
C = re[None, :] + 1j * im[:, None]
z = np.zeros_like(C)
bounded = np.ones(C.shape, dtype=bool)
for _ in range(80):
    z[bounded] = z[bounded] ** 2 + C[bounded]
    bounded &= np.abs(z) <= 2
map_ax.imshow(bounded, extent=[re[0], re[-1], im[0], im[-1]], origin="lower", cmap="Greys")
label_offsets = [(0.1, -0.35), (0.0, -0.45), (-0.4, 0.3), (-0.6, 0.15), (-0.5, -0.3), (0.0, 0.35)]
for c, (dx, dy) in zip(parameters, label_offsets):
    map_ax.plot(c.real, c.imag, "o", markersize=8, markeredgecolor="white", color="tab:red")
    map_ax.annotate(f"{c.real:g}{c.imag:+g}i", xy=(c.real, c.imag), xytext=(c.real + dx, c.imag + dy),
                    fontsize=8, color="tab:red", arrowprops=dict(arrowstyle="-", color="tab:red", linewidth=0.8),
                    bbox=dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.8))
map_ax.set_aspect("equal")
map_ax.set_xlabel("Re c")
map_ax.set_ylabel("Im c")
map_ax.set_title("Parameters on M")
plt.tight_layout()
plt.show()

### 5.6 Crossing the boundary of $M$ along the real axis

Moving $c$ along the real axis through the cusp at $\frac14$: for $c = 0.2$ and $c = 0.25$ the filled Julia set is a solid connected region; just past the cusp, at $c = 0.26$ and $c = 0.3$, the black interior is gone and only a dust-like set remains (it has no interior and no area; the colouring shows how long points linger near it).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def escape_time(z0, c, max_iter):
    z = z0.copy()
    counts = np.full(z.shape, max_iter)
    active = np.ones(z.shape, dtype=bool)
    for n in range(max_iter):
        z[active] = z[active] ** 2 + c
        escaped = active & (np.abs(z) > 2)
        counts[escaped] = n + 1
        active &= ~escaped
    return counts


grid = np.linspace(-1.5, 1.5, 400)
Z0 = grid[None, :] + 1j * grid[:, None]
cmap = plt.get_cmap("plasma").copy()
cmap.set_bad("black")

fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
for ax, c in zip(axes, [0.2, 0.25, 0.26, 0.3]):
    counts = escape_time(Z0, c, 300)
    ax.imshow(np.ma.masked_where(counts == 300, np.log(counts)), extent=[-1.5, 1.5, -1.5, 1.5], origin="lower", cmap=cmap)
    ax.set_title(f"c = {c}: black area = {np.mean(counts == 300) * 9:.3f}")
    ax.set_aspect("equal")
    ax.set_xlabel("Re z")
    ax.set_ylabel("Im z")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

1. Compute the orbit of $0$ for $c = -2$. Is $-2 \in M$?
2. Compute $z_1, \dots, z_5$ for $c = 0.5$ and decide whether $0.5 \in M$, citing the escape lemma.
3. Show that the filled Julia set of $c = 0$ is the closed unit disk.

### Intermediate

4. **Draw it first, then compute.** Sketch the main cardioid (cusp at $\frac14$, left end at $-\frac34$, top near $\frac14 + \frac i2$) and the disk $|c + 1| < \frac14$. Mark $c = -0.5$, $c = -1.1$ and $c = 0.3 + 0.6i$. Predict which are in $M$ and why. Then check by iterating.
5. For $c = 0.2$, find both fixed points of $f_c$ and their multipliers. Iterate from $0$ and confirm convergence to the attracting one.
6. Prove that $M$ is symmetric about the real axis.

### Advanced

7. Starting from $f_c(z^*) = z^*$ and $|f_c'(z^*)| = 1$, derive the cardioid $c(t) = \frac{e^{it}}{2} - \frac{e^{2it}}{4}$, and find its intersections with the real axis.
8. Derive the period-2 condition $|c + 1| < \frac14$ by computing $f_c(f_c(z)) - z$, factoring out $f_c(z) - z$, and computing the multiplier.
9. Show that for real $c > \frac14$ the real orbit of $0$ is strictly increasing and tends to $+\infty$, so $M \cap \mathbb{R} \subset [-2, \frac14]$. (With Exercise 1 and the cusp, $M \cap \mathbb{R} = [-2, \frac14]$.)

## 7. Solutions

1. $0 \to -2 \to 2 \to 2 \to 2 \to \cdots$: the orbit lands on the fixed point $2$ and stays, so it is bounded and $-2 \in M$ (the left-most point of $M$, the tip of the "antenna").
2. $z_1 = 0.5$, $z_2 = 0.75$, $z_3 = 1.0625$, $z_4 = 1.6289$, $z_5 = 3.1533$. Since $|z_5| > 2 \ge |c|$, the lemma says the orbit escapes: $0.5 \notin M$.
3. $f_0^n(z_0) = z_0^{2^n}$ has modulus $|z_0|^{2^n}$, bounded iff $|z_0| \le 1$.
4. Sketch: $-0.5$ is on the real axis between $-\frac34$ and $\frac14$, inside the cardioid; $-1.1$ is within $0.1$ of $-1$, inside the period-2 disk; $0.3 + 0.6i$ lies to the right of and above the cardioid's top, outside both. Prediction: the first two are in $M$, the third probably not. Check: for $-0.5$, $|\mu| = |1 - \sqrt{3}| = 0.732 < 1$; for $-1.1$, $|c + 1| = 0.1 < 0.25$; for $0.3 + 0.6i$ the orbit exceeds $2$ after 15 iterations.
5. $z^* = 0.2764$ (multiplier $0.553$, attracting) and $z^* = 0.7236$ (multiplier $1.447$, repelling). The orbit of $0$ converges to $0.2764$, with the error shrinking by about $0.553$ per step.
6. Conjugating, $\overline{z_{n+1}} = \overline{z_n}^2 + \bar c$, so the orbit of $0$ under $f_{\bar c}$ is the conjugate of the orbit under $f_c$ and has the same moduli. Hence $c \in M \iff \bar c \in M$.
7. With $\mu = 2z^*$, the fixed-point equation gives $c = \frac{\mu}{2} - \frac{\mu^2}{4}$; neutral multiplier $|\mu| = 1$ means $\mu = e^{it}$. On the real axis, $\mathrm{Im}\,c(t) = \frac{\sin t}{2} - \frac{\sin 2t}{4} = \frac{\sin t}{2}(1 - \cos t) = 0$ iff $t = 0$ or $\pi$, giving $c = \frac14$ and $c = -\frac34$.
8. $f_c(f_c(z)) - z = (z^2 + c)^2 + c - z = (z^2 - z + c)(z^2 + z + c + 1)$ (expand to check). The second factor has roots $z_1, z_2$ with $z_1z_2 = c + 1$, and the multiplier is $(f_c^{\circ 2})'(z_1) = f_c'(z_2)f_c'(z_1) = 4z_1z_2 = 4(c + 1)$; it is attracting iff $|c + 1| < \frac14$.
9. $z_{n+1} - z_n = z_n^2 - z_n + c = \left(z_n - \frac12\right)^2 + c - \frac14 > 0$ for real $z_n$ when $c > \frac14$, in fact $\ge c - \frac14$. So the orbit increases by at least $c - \frac14 > 0$ per step and tends to $+\infty$.

Numerical checks:

In [ ]:
import numpy as np
import sympy as sp


def orbit(c, steps):
    z, points = 0j, []
    for _ in range(steps):
        z = z * z + c
        points.append(z)
    return points


print("Ex 1:", [p.real for p in orbit(-2, 6)])
print("Ex 2:", [round(p.real, 4) for p in orbit(0.5, 5)])
for c in [-0.5, -1.1, 0.3 + 0.6j]:
    z, escaped_at = 0j, None
    for n in range(1, 1001):
        z = z * z + c
        if abs(z) > 2:
            escaped_at = n
            break
    print(f"Ex 4: c = {c}: escaped after {escaped_at} iterations" if escaped_at else f"Ex 4: c = {c}: bounded for 1000 iterations")
print("Ex 5: orbit of 0 after 40 steps:", round(orbit(0.2, 40)[-1].real, 6), " fixed points:", np.round(np.roots([1, -1, 0.2]), 4))
print("Ex 6:", np.allclose(np.abs(orbit(0.3 + 0.4j, 8)), np.abs(orbit(0.3 - 0.4j, 8))))
z, c = sp.symbols("z c")
print("Ex 8:", sp.factor(sp.expand((z**2 + c) ** 2 + c - z)))
print("Ex 9: c = 0.26, first increments:", np.round(np.diff([0] + [p.real for p in orbit(0.26, 8)]), 4))

## 8. Conclusion

- One step of $z \mapsto z^2 + c$ squares the modulus, doubles the argument and shifts by $c$; iterating it produces the Julia sets $K_c$ (vary the start) and the Mandelbrot set $M$ (start at the critical point $0$, vary $c$).
- **Escape lemma:** once $|z_n| > \max(2, |c|)$ the orbit runs off to infinity, so $c \in M \iff |z_n| \le 2$ for all $n$, and $M$ lies in the disk $|c| \le 2$. This makes escape-time pictures computable.
- Vectorized numpy with an "active" mask renders an $800 \times 700$ image in a few seconds.
- Attracting cycles explain the large components of $M$: the main cardioid $c = \frac{\mu}{2} - \frac{\mu^2}{4}$ ($|\mu| < 1$) for fixed points and the disk $|c + 1| < \frac14$ for 2-cycles.
- **Dichotomy:** $K_c$ is connected for $c \in M$ and a Cantor dust for $c \notin M$ — the Mandelbrot set is the catalogue of connected Julia sets.

**Next:** notebook 06 closes the series with physics where complex numbers are not a convenience but the substance: a quantum bit's state is a pair of complex amplitudes, and *phases* — invisible to a single measurement — decide the outcome of interference.